<a href="https://colab.research.google.com/github/Spartaco17/SLY-Assistant/blob/main/Notebooks/colab_ebook2audiobook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Welcome to the ebook2audiobook Google Colab!
## Features
- 🔧 **TTS Engines supported**: XTTSv2, Bark, Fairseq, VITS, Tacotron2, Tortoise, GlowTTS, YourTTS- 📚 **Convert multiple file formats**: .epub, .mobi, .azw3, .fb2, .lrf, .rb, .snb, .tcr, .pdf, .txt, .rtf, .doc, .docx, .html, .odt, .azw, .tiff, .tif, .png, .jpg, .jpeg, .bmp
- 🔍 **OCR scanning** for files with text pages as images
- 🔊 **High-quality text-to-speech** from near realtime to near real voice
- 🗣️ **Optional voice cloning** using your own voice file
- 🌐 **Supports 1158 languages** ([supported languages list](https://dl.fbaipublicfiles.com/mms/tts/all-tts-languages.html))
- 💻 **Low-resource friendly** — runs on **2 GB RAM / 1 GB VRAM (minimum)**
- 🎵 **Audiobook output formats**: mono or stereo aac, flac, mp3, m4b, m4a, mp4, mov, ogg, wav, webm
- 🧠 **SML tags supported** — fine-grained control of breaks, pauses, voice switching and more ([see below](#sml-tags-available))
- 🧩 **Optional custom model** using your own trained model (XTTSv2 only, other on request)
- 🎛️ **Fine-tuned preset models** trained by the E2A Team<br/>
     <i>(Contact us if you need additional fine-tuned models, or if you'd like to share yours to the official preset list)</i>
## Want to run locally for free? ⬇
## [Check out the ebook2audiobook github!](https://github.com/DrewThomasson/ebook2audiobook)

In [8]:
import os
import subprocess
import time
import sys
import re

# Emojis for logs
CHECK_MARK = "✅"
CROSS_MARK = "❌"

SCRIPT_DIR = "/content/ebook2audiobook"

def run_command_with_log(command, description):
    print(f"\n> {description}...")
    start_time = time.time()
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    stdout, stderr = process.communicate()
    if process.returncode != 0:
        print(f"{CROSS_MARK} Errore: {stderr.decode()}")
        return False
    print(f"{CHECK_MARK} Completato in {time.time() - start_time:.2f}s")
    return True

# --- Step 0: Git Repository ---
print("--- Configurazione Repository ---")
if not os.path.exists(SCRIPT_DIR):
    run_command_with_log(f"git clone --depth=1 https://github.com/DrewThomasson/ebook2audiobook.git {SCRIPT_DIR}", "Cloning repository")
else:
    run_command_with_log(f"cd {SCRIPT_DIR} && git reset --hard HEAD && git pull", "Aggiornamento e pulizia repository")

# --- Step 1: Patching Locale ---
print("\n--- Applicazione Patch Finali ---")
patch_file = os.path.join(SCRIPT_DIR, "lib/classes/device_installer.py")
app_file = os.path.join(SCRIPT_DIR, "app.py")

# 1. Patch Device Installer (Hardware logic & Torch version)
if os.path.exists(patch_file):
    with open(patch_file, "r") as f: c = f.read()
    c = re.sub(r"\s*@staticmethod\s*def.*?_normalize_version.*?\n(\s+.*?\n)*", "\n", c)
    sm = "\n    @staticmethod\n    def _normalize_version(v):\n        import re as _re_fix\n        if isinstance(v, list): return v\n        v_str = str(v)\n        return [int(x) for x in _re_fix.sub(r'[^0-9.]', '', v_str).split('.') if x]\n"
    if "class DeviceInstaller" in c: c = re.sub(r"(class DeviceInstaller.*?\:)", r"\1" + sm, c)
    c = c.replace("v <= version", "self._normalize_version(v) <= self._normalize_version(version)")
    c = c.replace("2.13.0", "2.11.0")
    c = re.sub(r"torch==[0-9.]+", "torch==2.11.0", c)
    with open(patch_file, "w") as f: f.write(c)

# 2. Patch App.py (Hard Bypass Environment Check)
if os.path.exists(app_file):
    with open(app_file, "r") as f: lines = f.readlines()
    with open(app_file, "w") as f:
        f.write("import sys\nsys.real_prefix = sys.prefix\n") # Injection iniziale
        skip = False
        for line in lines:
            # Rileva l'inizio del blocco di controllo venv/conda
            if "if not hasattr(sys, 'real_prefix')" in line or "if 'conda' in sys.version.lower()" in line:
                f.write("if False: # Bypassed by Colab Agent\n")
                skip = True
                continue
            # Se siamo dentro il blocco che chiama sys.exit(), saltiamolo
            if skip and "sys.exit()" in line:
                f.write("    pass\n")
                if line.strip() == "sys.exit()": skip = False
                continue
            if skip and "***********" in line and lines.index(line) > 50: # fine blocco msg
                 skip = False
            f.write(line)

print(f"{CHECK_MARK} Patch applicate.")

# --- Step 2: Env Setup ---
os.makedirs(os.path.join(SCRIPT_DIR, "tmp"), exist_ok=True)
os.makedirs(os.path.join(SCRIPT_DIR, "models"), exist_ok=True)
os.environ.update({"TMPDIR": os.path.join(SCRIPT_DIR, "tmp"), "TTS_CACHE": os.path.join(SCRIPT_DIR, "models")})

# --- Step 3: Launch ---
print("\n--- Avvio Gradio ---")
try:
    # Usiamo python3 esplicitamente per l'ambiente Colab
    get_ipython().system(f"cd {SCRIPT_DIR} && python3 app.py --script_mode native --share")
except Exception as e:
    print(f"{CROSS_MARK} Errore durante il lancio: {e}")

--- Configurazione Repository ---

> Aggiornamento e pulizia repository...
✅ Completato in 0.44s

--- Applicazione Patch Finali ---
✅ Patch applicate.

--- Avvio Gradio ---
***********
Wrong launch! ebook2audiobook must run in its own virtual environment!
NOTE: If you are running a Docker so you are probably using an old version of ebook2audiobook.
To solve this issue go to download the new version at https://github.com/DrewThomasson/ebook2audiobook
If the directory python_env does not exist in the ebook2audiobook root directory,
run your command with "./ebook2audiobook.command" for Linux and Mac or "ebook2audiobook.cmd" for Windows
to install it all automatically.

After the first run, you are free to use your command line with:
# go into ebook2audiobook folder then:
----------------------------------
# Linux/Mac
ebook2audiobook.sh [options]
# Windows
ebook2audiobook.cmd [options]
----------------------------------
or
----------------------------------
# Linux/Mac
conda activate ./pyt